# Enhancing Financial Market Predictions with Financial News Sentiment

## Project topic

This project investigates whether financial news sentiment is related to short-term stock market returns for selected technology companies.

The analysis combines two independent data sources:

1. **Financial news headlines** from `analyst_ratings_processed.csv`
2. **Historical stock market prices** from `tech_stock_prices_2020_to_today.csv`


---

## Research question

**Can daily financial news sentiment help explain or predict daily stock returns?**

We will focus on three companies that are present in both datasets and have sufficient news coverage:

- **AAPL** — Apple
- **TSLA** — Tesla
- **NVDA** — Nvidia


## Mathematical background

### Daily return

Daily return measures the relative change in closing price between two consecutive trading days:

$$
R_t = \frac{P_t - P_{t-1}}{P_{t-1}}
$$

where:

- $P_t$ is the closing price on day $t$
- $P_{t-1}$ is the closing price on the previous trading day

### Sentiment score

Each headline is converted into a numerical sentiment score:

$$
S_i \in [-1, 1]
$$

where:

- negative values indicate negative sentiment
- values close to zero indicate neutral sentiment
- positive values indicate positive sentiment

Daily sentiment is calculated as the average sentiment of all headlines for a company on a given day:

$$
\bar{S}_{t} = \frac{1}{n}\sum_{i=1}^{n} S_i
$$

### Correlation

The Pearson correlation coefficient measures the linear relationship between sentiment and returns:

$$
\rho_{S,R} = \frac{Cov(S, R)}{\sigma_S \sigma_R}
$$

### Regression

A simple regression model can be written as:

$$
R_t = \beta_0 + \beta_1 S_t + \epsilon_t
$$

where:

- $R_t$ is the stock return
- $S_t$ is the sentiment score
- $\beta_0$ is the intercept
- $\beta_1$ measures the relationship between sentiment and returns
- $\epsilon_t$ is the error term

In [16]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import re
import math
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)

## 1. Data loading

The notebook expects the following files to be available in the same folder as the notebook:

- `analyst_ratings_processed.csv`
- `tech_stock_prices_2020_to_today.csv`

If the notebook is executed in the original working environment, it will also try `/mnt/data`.

In [17]:
DATA_DIR = Path(".")
# if not (DATA_DIR / "analyst_ratings_processed.csv").exists():
#     DATA_DIR = Path("/mnt/data")

NEWS_PATH = DATA_DIR / "analyst_ratings_processed.csv"
STOCK_PATH = DATA_DIR / "tech_stock_prices_2020_to_today.csv"

print("News dataset path:", NEWS_PATH)
print("Stock dataset path:", STOCK_PATH)

News dataset path: analyst_ratings_processed.csv
Stock dataset path: tech_stock_prices_2020_to_today.csv


In [18]:
news_raw = pd.read_csv(
    NEWS_PATH,
    usecols=["title", "date", "stock"]
)

stock_raw = pd.read_csv(STOCK_PATH)

print("News shape:", news_raw.shape)
print("Stock shape:", stock_raw.shape)

display(news_raw.head())
display(stock_raw.head())

News shape: (1400466, 3)
Stock shape: (23160, 18)


,title,date,stock
0,Stocks That Hit 52-Week Highs On Friday,2020-06-05 10:30:00-04:00,A
1,Stocks That Hit 52-Week Highs On Wednesday,2020-06-03 10:45:00-04:00,A
2,71 Biggest Movers From Friday,2020-05-26 04:30:00-04:00,A
3,46 Stocks Moving In Friday's Mid-Day Session,2020-05-22 12:45:00-04:00,A
4,B of A Securities Maintains Neutral on Agilent...,2020-05-22 11:38:00-04:00,A


,index,Date,Open,High,Low,Close,Adj Close,Volume,Ticker,Dividends,Stock Splits,P/E Ratio,Market Cap,Price/Sales Ratio,Price/Book Ratio,Dividend Yield,Daily Return,20-Day MA
0,0,2020-01-02,74.059998,75.150002,73.797501,75.087502,72.960464,135480400,AAPL,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,2020-01-03,74.287498,75.144997,74.125000,74.357498,72.251122,146322800,AAPL,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,2020-01-06,73.447502,74.989998,73.187500,74.949997,72.826859,118387200,AAPL,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,2020-01-07,74.959999,75.224998,74.370003,74.597504,72.484352,108872000,AAPL,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,2020-01-08,74.290001,76.110001,74.290001,75.797501,73.650352,132079200,AAPL,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
